# A streaming RAG endpoint, end to end

This notebook runs the programs of the lesson *Build a streaming RAG endpoint end to end*. Run the cells in order. No GPU and no API key are needed (the keys in the code are invented demo values). The retrieval and the model are a stand-in that returns fixed pieces. One section starts a server inside the notebook on 127.0.0.1 port 8892 and stops it again. The outputs saved in the cells come from a real run with the pinned versions.

In [ ]:
!pip install -q fastapi==0.141.1 starlette==1.6.0 httpx==0.28.1 uvicorn==0.53.0 sse-starlette==3.7.1
print('installed')

## 1. The stand-in model, the event writer and the shortest endpoint

In [ ]:
SOURCES = [{"n": 1, "doc": "leave-policy"}, {"n": 2, "doc": "product-faq"}]
# STAND-IN for the model: the pieces it "generates", one at a time (note the marker [1] split over two pieces)
PIECES = ["Full-time ", "employees ", "receive ", "24 days ", "of paid ", "annual ", "leave ", "[", "1]."]

def model_pieces(fail_after=None):
    for i, piece in enumerate(PIECES):
        if i == fail_after:
            raise ConnectionError("the model provider closed the connection")
        yield piece

import json

def sse(event, data, event_id=None):
    """one event as text: field lines, then an empty line that ends the event"""
    lines = [f"event: {event}"]
    if event_id is not None:
        lines.append(f"id: {event_id}")
    lines.append("data: " + json.dumps(data, ensure_ascii=False))     # JSON escapes newlines, so one line
    return "\n".join(lines) + "\n\n"

def answer_events(pieces, request_id="req-0001"):
    """the stream of the contract: sources, tokens, then exactly one of done or error"""
    yield sse("sources", {"request_id": request_id, "sources": SOURCES})
    count = 0
    try:
        for piece in pieces:
            yield sse("token", {"text": piece})
            count += 1
    except ConnectionError:
        yield sse("error", {"code": "upstream_unavailable", "message": "a dependency is not answering",
                            "retryable": True, "request_id": request_id})
        return
    yield sse("done", {"request_id": request_id, "finish_reason": "stop", "tokens": count})

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

first = FastAPI()

@first.post("/v1/ask/stream")
def first_stream():
    return StreamingResponse(answer_events(model_pieces()), media_type="text/event-stream")

r = TestClient(first).post("/v1/ask/stream")
print(r.status_code, r.headers["content-type"])
events = r.text.split("\n\n")[:-1]
print(len(events), "events")
for event in events[:2] + events[-1:]:
    print(event.replace("\n", " | "))

200 text/event-stream; charset=utf-8
11 events
event: sources | data: {"request_id": "req-0001", "sources": [{"n": 1, "doc": "leave-policy"}, {"n": 2, "doc": "product-faq"}]}
event: token | data: {"text": "Full-time "}
event: done | data: {"request_id": "req-0001", "finish_reason": "stop", "tokens": 9}


## 2. Does the test client stream?

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

probe = FastAPI()
order = []

@probe.get("/three")
def three():
    def events():
        for n in (1, 2, 3):
            order.append(f"made {n}")
            yield f"data: {n}\n\n"
    return StreamingResponse(events(), media_type="text/event-stream")

with TestClient(probe).stream("GET", "/three") as r:
    for line in r.iter_lines():
        if line:
            order.append("got " + line[-1])
print(", ".join(order))

made 1, made 2, made 3, got 1, got 2, got 3


## 3. The service: contract errors, asynchronous model, relay and endpoint

In [ ]:
ERRORS = {  # code: (status line, may the caller retry the same request?)
    "invalid_json": ("400 Bad Request", False),
    "unauthorized": ("401 Unauthorized", False),
    "not_found": ("404 Not Found", False),
    "method_not_allowed": ("405 Method Not Allowed", False),
    "payload_too_large": ("413 Content Too Large", False),
    "unsupported_media_type": ("415 Unsupported Media Type", False),
    "invalid_request": ("422 Unprocessable Content", False),
    "rate_limited": ("429 Too Many Requests", True),
    "internal_error": ("500 Internal Server Error", True),
    "upstream_unavailable": ("503 Service Unavailable", True),
    "deadline_exceeded": ("504 Gateway Timeout", True),
}

class ApiError(Exception):
    def __init__(self, code, message, details=None, headers=()):
        super().__init__(message)
        self.code, self.message, self.details, self.headers = code, message, details, list(headers)

def error_body(error, request_id):
    body = {"code": error.code, "message": error.message, "retryable": ERRORS[error.code][1], "request_id": request_id}
    if error.details:
        body["details"] = error.details
    return {"error": body}

def validate(data):
    """returns (clean request, problems); every problem names its field"""
    if not isinstance(data, dict):
        return None, [{"field": "body", "problem": "must be a JSON object"}]
    problems = []
    def bad(field, problem):
        problems.append({"field": field, "problem": problem})
    question = data.get("question")
    if question is None:
        bad("question", "is required")
    elif not isinstance(question, str):
        bad("question", "must be a string")
    elif not question.strip():
        bad("question", "must not be empty")
    elif len(question) > 500:
        bad("question", "must be at most 500 characters")
    top_k = data.get("top_k", 4)
    if type(top_k) is not int:            # type(), because True would pass isinstance(top_k, int)
        bad("top_k", "must be an integer")
    elif not 1 <= top_k <= 10:
        bad("top_k", "must be from 1 to 10")
    for name in sorted(set(data) - {"question", "top_k"}):
        bad(name, "unknown field")
    if problems:
        return None, problems
    return {"question": question.strip(), "top_k": top_k}, []

In [ ]:
import asyncio

async def standin_model(fail_after=None):
    """the same stand-in pieces, as an asynchronous generator: it waits without blocking the server"""
    for piece in model_pieces(fail_after):
        await asyncio.sleep(0)
        yield piece

async def with_silences(model, every):
    """passes the model's pieces on; gives None each time `every` seconds go by without one"""
    task = None
    try:
        while True:
            task = task or asyncio.ensure_future(anext(model))
            done, _ = await asyncio.wait({task}, timeout=every)
            if not done:
                yield None                      # the model is silent; the same task keeps waiting
                continue
            try:
                piece = task.result()
            except StopAsyncIteration:
                return
            task = None
            yield piece
    finally:
        if task and not task.done():
            task.cancel()                       # stops the model while it is waiting
        else:
            await model.aclose()                # or while it is paused between two pieces

In [ ]:
import re

async def relay(model, request_id, log, every):
    """the stream of the contract, with one log record however it ends"""
    source = with_silences(model, every)
    sent, reason, text, cited = 0, "client_closed", "", set()
    docs = {s["n"]: s["doc"] for s in SOURCES}
    try:
        yield sse("sources", {"request_id": request_id, "sources": SOURCES})
        async for piece in source:
            if piece is None:
                yield ": keep-alive\n\n"
                continue
            sent += 1
            text += piece
            for n in map(int, re.findall(r"\[(\d+)\]", text)):      # whole markers only
                if n in docs and n not in cited:
                    cited.add(n)
                    yield sse("citation", {"n": n, "doc": docs[n]})
            yield sse("token", {"text": piece})
        reason = "stop"
        yield sse("done", {"request_id": request_id, "finish_reason": "stop", "tokens": sent})
    except ConnectionError:
        reason = "error"
        yield sse("error", {"code": "upstream_unavailable", "message": "a dependency is not answering",
                            "retryable": True, "request_id": request_id})
    finally:
        await source.aclose()
        log({"request_id": request_id, "finish_reason": reason, "tokens": sent})

In [ ]:
import math
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

KEYS = {"demo-key-acme": "acme", "demo-key-zen": "zen"}      # invented demo keys: API key -> tenant

class ClosingStream(StreamingResponse):
    """close the generator as soon as the response is over, also when the client left"""
    async def __call__(self, scope, receive, send):
        try:
            await super().__call__(scope, receive, send)
        finally:
            await self.body_iterator.aclose()

def make_api(model=standin_model, ready=lambda: True, limiter=None, log=lambda record: None, every=15.0):
    api = FastAPI(redirect_slashes=False, docs_url=None, redoc_url=None, openapi_url=None)
    counter = [0]

    @api.exception_handler(ApiError)                         # problems found before the stream starts
    async def refused(request, e):
        counter[0] += 1
        rid = f"req-{counter[0]:04d}"
        return JSONResponse(error_body(e, rid), status_code=int(ERRORS[e.code][0][:3]),
                            headers={**dict(e.headers), "X-Request-ID": rid})

    @api.post("/v1/ask/stream")
    async def ask_stream(request: Request):
        scheme, _, key = request.headers.get("authorization", "").partition(" ")
        if scheme != "Bearer" or key not in KEYS:
            raise ApiError("unauthorized", "missing or unknown API key", headers=[("WWW-Authenticate", "Bearer")])
        if limiter:
            allowed, wait = limiter(KEYS[key])
            if not allowed:
                raise ApiError("rate_limited", "too many requests", headers=[("Retry-After", str(math.ceil(wait)))])
        if request.headers.get("content-type", "").split(";")[0].strip() != "application/json":
            raise ApiError("unsupported_media_type", "send application/json")
        raw = await request.body()
        if len(raw) > 4096:
            raise ApiError("payload_too_large", "the body is larger than 4096 bytes")
        try:
            data = json.loads(raw)
        except ValueError:
            raise ApiError("invalid_json", "the body is not valid JSON")
        wanted, problems = validate(data)
        if problems:
            raise ApiError("invalid_request", "the request does not match the schema", problems)
        if not ready():                                      # last check that can still change the status
            raise ApiError("upstream_unavailable", "a dependency is not answering", headers=[("Retry-After", "5")])
        counter[0] += 1
        rid = f"req-{counter[0]:04d}"
        return ClosingStream(relay(model(), rid, log, every), media_type="text/event-stream",
                             headers={"X-Request-ID": rid, "Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

    return api

## 4. The contract, checked inside the process

In [ ]:
from fastapi.testclient import TestClient

records = []
KEY = {"Authorization": "Bearer demo-key-acme"}
ASK = {"question": "How much annual leave do I get?"}

def ask(api, body=ASK, headers=KEY):
    return TestClient(api).post("/v1/ask/stream", json=body, headers=headers)

def show(label, r):
    code = r.json()["error"]["code"]
    print(f"{label:14} {r.status_code} {r.headers['content-type']} {code}")

show("no key", ask(make_api(), headers={}))
show("empty question", ask(make_api(), body={"question": " "}))
show("over the limit", ask(make_api(limiter=lambda tenant: (False, 1.5))))
show("provider down", ask(make_api(ready=lambda: False)))

def names(text):
    return [line[7:] for line in text.split("\n") if line.startswith("event: ")]

good = ask(make_api(log=records.append))
print("complete:", good.status_code, names(good.text))
broken = ask(make_api(model=lambda: standin_model(fail_after=2), log=records.append))
print("broken:  ", broken.status_code, names(broken.text))
for record in records:
    print("log:", record["finish_reason"], record["tokens"])
without = "".join(part + "\n\n" for part in good.text.split("\n\n")[:-1] if not part.startswith("event: citation"))
print("same text as the reference writer:", without == "".join(answer_events(model_pieces())))

no key         401 application/json unauthorized
empty question 422 application/json invalid_request
over the limit 429 application/json rate_limited
provider down  503 application/json upstream_unavailable
complete: 200 ['sources', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'citation', 'token', 'done']
broken:   200 ['sources', 'token', 'token', 'error']
log: stop 9
log: error 2
same text as the reference writer: True


## 5. A real server: early delivery, a heartbeat and a client that leaves

The server runs in a second thread, which has its own event loop, and is stopped at the end of the cell. Port 8892 must be free.

In [ ]:
class SSEParser:
    def __init__(self):
        self.buffer = ""            # characters of a line that is not finished yet
        self.event, self.data = "", []
        self.last_id = ""

    def feed(self, chunk):
        out = []
        self.buffer += chunk
        while True:
            ends = [i for i in (self.buffer.find("\n"), self.buffer.find("\r")) if i >= 0]
            if not ends:
                return out
            at = min(ends)
            if self.buffer[at] == "\r" and at + 1 == len(self.buffer):
                return out          # a "\n" may still follow: wait
            size = 2 if self.buffer[at:at + 2] == "\r\n" else 1
            line, self.buffer = self.buffer[:at], self.buffer[at + size:]
            self._line(line, out)

    def _line(self, line, out):
        if line == "":              # empty line: the event is complete
            if self.data:
                out.append({"event": self.event or "message", "data": "\n".join(self.data), "id": self.last_id})
            self.event, self.data = "", []
            return
        if line.startswith(":"):    # a comment, used as a heartbeat
            return
        name, _, value = line.partition(":")
        if value.startswith(" "):
            value = value[1:]
        if name == "event":
            self.event = value
        elif name == "data":
            self.data.append(value)
        elif name == "id":
            self.last_id = value

In [ ]:
import threading
import time
import httpx
import uvicorn

def start(api, port=8892):
    server = uvicorn.Server(uvicorn.Config(api, host="127.0.0.1", port=port, log_level="critical"))
    thread = threading.Thread(target=server.run)
    thread.start()
    while not server.started:
        assert thread.is_alive(), "the server did not start (is the port in use?)"
        time.sleep(0.01)
    return server, thread

def stop(server, thread):
    server.should_exit = True
    thread.join(10)
    return not thread.is_alive()

In [ ]:
release, model_closed, logged = threading.Event(), threading.Event(), threading.Event()
records = []

def keep(record):
    records.append(record)
    logged.set()

async def pausing_model():                      # one piece, then silence until the client says "go on"
    yield PIECES[0]
    while not release.is_set():
        await asyncio.sleep(0.01)
    for piece in PIECES[1:]:
        yield piece

async def endless_model():                      # never finishes by itself
    try:
        while True:
            await asyncio.sleep(0.01)
            yield "word "
    finally:
        model_closed.set()

chosen = [pausing_model]
server, thread = start(make_api(model=lambda: chosen[0](), log=keep, every=0.05))
URL = "http://127.0.0.1:8892/v1/ask/stream"
POST = dict(json={"question": "annual leave?"}, headers={"Authorization": "Bearer demo-key-acme"}, timeout=5)
try:
    parser, names, text, early, beat = SSEParser(), [], "", False, False
    with httpx.stream("POST", URL, **POST) as r:
        for chunk in r.iter_text():
            for e in parser.feed(chunk):
                names.append(e["event"])
                if e["event"] == "token":
                    text += json.loads(e["data"])["text"]
                    early = early or not release.is_set()
            if "token" in names and ": keep-alive" in chunk:
                beat = True
                release.set()
    print("first token arrived while the model was unfinished:", early)
    print("heartbeat arrived during the silence:", beat)
    print(names[-1], "|", text)

    chosen[0] = endless_model
    logged.clear()
    parser = SSEParser()
    with httpx.stream("POST", URL, **POST) as r:
        for chunk in r.iter_text():
            if any(e["event"] == "token" for e in parser.feed(chunk)):
                break                           # leave the block: the connection is closed
    print("the client left after a token")
    print("model closed:", model_closed.wait(5))
    print("log written:", logged.wait(5), records[-1]["finish_reason"], "| tokens counted:", records[-1]["tokens"] > 0)
finally:
    release.set()
    print("server stopped:", stop(server, thread))

first token arrived while the model was unfinished: True
heartbeat arrived during the silence: True
done | Full-time employees receive 24 days of paid annual leave [1].
the client left after a token
model closed: True
log written: True client_closed | tokens counted: True
server stopped: True


## 6. The helpers: fastapi.sse and sse-starlette

In [ ]:
import json
from fastapi import Depends, FastAPI, HTTPException
from fastapi.sse import EventSourceResponse, ServerSentEvent
from fastapi.testclient import TestClient
import sse_starlette

helpers = FastAPI()
DATA = {"text": "Zürich "}

def refuse():
    raise HTTPException(401, "unknown key")

@helpers.get("/fastapi", response_class=EventSourceResponse)
def with_fastapi():                                          # a generator endpoint: you yield events
    yield ServerSentEvent(data=DATA, event="token", id="1")

@helpers.get("/package")
async def with_package():
    async def events():
        yield {"event": "token", "id": "1", "data": json.dumps(DATA, ensure_ascii=False)}
        yield {"event": "token", "data": DATA}               # a dict as data, not turned into JSON
    return sse_starlette.EventSourceResponse(events())

@helpers.get("/early", response_class=EventSourceResponse, dependencies=[Depends(refuse)])
def refused_early():
    yield {}

@helpers.get("/late", response_class=EventSourceResponse)
def refused_late():
    raise HTTPException(401, "unknown key")                  # inside the generator body
    yield {}

client = TestClient(helpers, raise_server_exceptions=False)
for path in ("/fastapi", "/package"):
    r = client.get(path)
    text = r.content.decode()
    end = "\r\n" if "\r\n" in text else "\n"
    print(path, "cache-control:", r.headers["cache-control"], "| line ending:", repr(end))
    for event in text.split(end + end)[:-1]:
        print("   " + "   ".join(f"[{line}]" for line in event.split(end)))
for path in ("/early", "/late"):
    r = client.get(path)
    print(path, r.status_code, r.text or "(empty body)")

/fastapi cache-control: no-cache | line ending: '\n'
   [event: token]   [data: {"text": "Z\u00fcrich "}]   [id: 1]
/package cache-control: no-store | line ending: '\r\n'
   [id: 1]   [event: token]   [data: {"text": "Zürich "}]
   [event: token]   [data: {'text': 'Zürich '}]
/early 401 {"detail":"unknown key"}
/late 200 (empty body)


## Try next

Make the endless model of section 5 a plain generator behind its own small endpoint, disconnect a client and look at when its `finally` block runs. Never paste a real secret into a notebook cell.